### Link Grabber


In [27]:
from selenium import webdriver
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import time
import random
import re

import pyktok as pyk

options = Options()
service = Service(ChromeDriverManager().install())


In [ ]:
def grab_tiktok_links(
        goal: int = 100,
        url: str = 'https://www.tiktok.com/tag/ai',
        a_class: str = 'css-1q1pv25-5e6d46e3--AMetaCaptionLine',
        wait_time: int = 1,
        ) -> list[str]:
    """
    Args:
        goal: Number of links seen before returning
        url: Tiktok hashtag url
        a_class: <a> container class for lookup
        wait_time: Seconds to pause between requesting more links

    Returns:
        links: Unique urls. Size not match goal.
    """
    driver = webdriver.Chrome(service=service, options=options)
    driver.get(url)

    wait = WebDriverWait(driver, timeout=2)
    html = driver.find_element(By.TAG_NAME,"html")

    number_of_links = 0

    while number_of_links < goal:
        wait.until(
            lambda d: len(d.find_elements(By.CLASS_NAME, a_class)) != number_of_links
        )

        source = driver.page_source
        soup = BeautifulSoup(source, "html.parser")
        a = soup.find_all("a", {"class":a_class})
        number_of_links = len(a)

        time.sleep(max(0, random.gauss(wait_time, 0.2)))

        html.send_keys(Keys.END)
    driver.quit()

    links = set()
    
    for vid in a:
        url = vid['href']
        links.add(url)

    print(f"Successfully grabbed {len(links)} urls.")
    
    return list(links)


In [ ]:
def save_links(
        links: list,
        path: Path = Path("csvs"),
        filename: str = "urls.csv",
        replace: bool = False
        ) -> None:
    """
    Args:
        path: Defaults to "csvs" folder in same dir
        replace: Whether to overwrite existing files sharing the filename. False will increment a counter suffix.
    """
    df = pd.DataFrame(links, columns=["url"])

    path.mkdir(parents=True, exist_ok=True)

    filepath = path / filename

    if not replace:
        stem = Path(filename).stem
        suffix = Path(filename).suffix

        count = 0

        while filepath.exists():
            filepath = path / f"{stem}_{count}{suffix}"
            count += 1

    df.to_csv(filepath, index=False)

In [ ]:
def save_video_from_url(
        url: str,
        path: Path = Path("data")
        ) -> None:
    """
    Args:
        path: Defaults to "data" folder in same dir
    """
    path.mkdir(parents=True, exist_ok=True)

    # Example URL: https://www.tiktok.com/@username/video/7589040432898657550

    username_match = re.search(r'@([^/]+)', url)
    username = username_match.group(1) if username_match else None

    video_id_match = re.search(r'/video/(\d+)', url)
    video_id = video_id_match.group(1) if video_id_match else None


    filepath = path / f"{username}_{video_id}"
    pyk.save_tiktok(url, True, filepath)
    pyk.save_tiktok()


def save_video_batch(
        links: list[str] = [],
        link_folder: Path = Path('csvs'),
        start: int = 0,
        goal: int = 10,
        wait: int = 0,
        path: Path = Path("data")
        ) -> None:
    
    """
    Args:
        links: List of video URLs. If none given, will search for them
        start: From what index to begin
        goal: How many videos to download before stopping
        wait: Seconds to wait between downloads
        path: Defaults to "data" folder in same dir
    """

    if not links:
        csv_files = list(link_folder.glob('*.csv'))
        
        if csv_files:
            dfs = [pd.read_csv(file) for file in csv_files]
            combined_df = pd.concat(dfs, ignore_index=True)
            
            links = combined_df['url'].tolist()
            print(f"Loaded {len(links)} URLs from {len(csv_files)} CSV files")
        else:
            print(f"No CSV files found in {link_folder}")

    if not links:
        print("No links found, and none could be imported.")
        return
    if start > len(links):
        print(f"Start index {start} is Out of range.")
        return
    end = min(len(links), start + goal)

    for i in tqdm(range(start, end), 
              desc="Downloading videos",
              unit="video"):
        save_video_from_url(links[i], path)
        time.sleep(wait)
    print(f"Videos saved to {path}.")


In [19]:
res = grab_tiktok_links(goal = 100)
save_links(res)

In [29]:
save_video_batch(goal = 3)

Loaded 110 URLs from 1 CSV files


Saved video
 https://www.tiktok.com/@egotthatwork91/video/7589040432898657550 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@egotthatwork91/video/7589040432898657550 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper


Saved video
 https://www.tiktok.com/@queenlibas.official/video/7596509238902525206 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@queenlibas.official/video/7596509238902525206 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper


Saved video
 https://www.tiktok.com/@dcneocdz328/video/7581360864889015566 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Saved metadata for video
 https://www.tiktok.com/@dcneocdz328/video/7581360864889015566 
to
 c:\Users\vi\Documents\GitHub\Video_detection_model-WIP-\scripts\scraper
Videos saved to data.
